# CampusIQ: Model Experimentation\nThis notebook experiments with different ML models for student attrition prediction.

In [ ]:
import pandas as pd\nimport numpy as np\nfrom sklearn.model_selection import cross_val_score, GridSearchCV\nfrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier\nfrom sklearn.linear_model import LogisticRegression\nimport xgboost as xgb\nfrom sklearn.metrics import roc_auc_score, classification_report\nimport mlflow\nfrom ml.models.train import load_data, engineer_features, prepare_data\n\nmlflow.set_tracking_uri('http://localhost:5000')\nmlflow.set_experiment('campusiq-model-comparison')

## Load and Prepare Data

In [ ]:
df = load_data()\ndf = engineer_features(df)\nX, y, feature_names = prepare_data(df)\n\nprint(f'Dataset shape: {X.shape}')\nprint(f'Target distribution:\n{y.value_counts()}')

## Model Comparison

In [ ]:
models = {\n    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),\n    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),\n    'GradientBoosting': GradientBoostingClassifier(random_state=42),\n    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='auc')\n}\n\nresults = {}\nfor name, model in models.items():\n    with mlflow.start_run(run_name=name):\n        scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')\n        results[name] = {\n            'mean_auc': scores.mean(),\n            'std_auc': scores.std()\n        }\n        mlflow.log_param('model', name)\n        mlflow.log_metric('cv_auc_mean', scores.mean())\n        mlflow.log_metric('cv_auc_std', scores.std())\n        print(f'{name}: {scores.mean():.4f} (+/- {scores.std()*2:.4f})')\n\n# Summary\nresults_df = pd.DataFrame(results).T\nresults_df.sort_values('mean_auc', ascending=False)

## Hyperparameter Tuning (XGBoost)

In [ ]:
param_grid = {\n    'max_depth': [4, 6, 8],\n    'learning_rate': [0.05, 0.1, 0.2],\n    'n_estimators': [100, 200, 300],\n}\n\nxgb_model = xgb.XGBClassifier(random_state=42, eval_metric='auc')\ngrid_search = GridSearchCV(xgb_model, param_grid, cv=3, scoring='roc_auc', n_jobs=-1)\ngrid_search.fit(X, y)\n\nprint(f'Best parameters: {grid_search.best_params_}')\nprint(f'Best CV AUC: {grid_search.best_score_:.4f}')